# Basic Imports and Cleaning

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("data/potential-talents - Aspiring human resources - seeking human resources.csv")

In [ ]:
df.head(25)

In [ ]:
df.info()

There are no null values aside from in fit, which we will be filling up with our calculated fit score.

### removing duplicates

In [ ]:
#checking for duplicates
df.duplicated(subset=['job_title', 'location', 'connection']).sum()

In [ ]:
#examining duplicates
df[df.duplicated(subset=['job_title', 'location', 'connection'])].sort_values(by='job_title')

In [ ]:
#removing duplicates
df = df.drop_duplicates(subset=['job_title', 'location', 'connection']).reset_index(drop=True)

In [ ]:
df.head(25)

In [ ]:
df.info()

Now that we've dropped duplicates that has cut the data set nearly in half. Next it's time to clean up the job titles to be consistent and the connections to be numeric. Let's check location for consistency as well.

### Data Transformation
cleaning up data types and job title consistency

In [ ]:
df['location'].value_counts()

In [ ]:
#removing the + from the connection column
df['connection'] = df['connection'].str.replace('+', '', regex=False)

In [ ]:
df.head(25)

In [ ]:
#Making connection numeric
df['connection'] = df['connection'].astype(int)

In [ ]:
df.info()

In [ ]:
df['job_title'].value_counts()

In [ ]:
df['job_title'] = df['job_title'].str.replace(r'\bHR\b', 'Human Resources', regex=True)
df['job_title'].value_counts()

I do see this one that is obviously a business looking for candidates. Since this is not what the business is looking for I will remove it

In [ ]:
df.head(53)

In [ ]:
#dropping nortia staffing advertisement at index 23
df = df.drop(23).reset_index(drop=True)

In [ ]:
df.head(52)

In [ ]:
#there appear to be some extra spaces in some of these.
df['job_title'] = df['job_title'].str.replace('  ', ' ', regex=True)

In [ ]:
#checking how many columns contain our basic keywords.
df['job_title'].str.contains('human resources', case=False, na=False).sum()

In [ ]:
df['job_title'].str.contains('aspiring human resources', case=False, na=False).sum()

In [ ]:
df['job_title'].str.contains('seeking human resources', case=False, na=False).sum()

In [ ]:
#looking back at the data there are a couple of people who would potentially be missed due to wording
df['job_title'] = df['job_title'].str.replace("seeking a position in Human Resources", "seeking human resources position", regex=True)
df['job_title'] = df['job_title'].str.replace("Seeking an Entry-Level Human Resources Position", "seeking human resources position", regex=True)

In [ ]:
df['job_title'].str.contains('seeking human resources', case=False, na=False).sum()

## additional EDA

In [ ]:
df["connection"].describe()

In [ ]:
plt.hist(df['connection'])

# Model

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

In [ ]:
def ranker(df, target_phrases=[], starred=[], alpha=0.5):
#creating the initial fit score
    target_phrases = target_phrases or []
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(df["job_title"])
    target_matrix = vectorizer.transform(target_phrases)
    similarity_scores = cosine_similarity(tfidf_matrix, target_matrix).max(axis=1)
    scaled_connections = MinMaxScaler(feature_range=(0, 1)).fit_transform(df[['connection']]).flatten()
    df["fit"] = (similarity_scores * 0.7) + (scaled_connections * .3)

#starring feature
    starred = starred or []
    if len(starred) > 0:
      starred_vectors = tfidf_matrix[starred]
      centroid = np.asarray(starred_vectors.mean(axis=0))
      sim_scores = cosine_similarity(tfidf_matrix, centroid).flatten()
      df["starred_fit"] = ((1 - alpha) * df['fit']) + (alpha * sim_scores)
    else:
      df["starred_fit"] = df['fit']
    df["starred_fit"] = df["starred_fit"].clip(0.0, 1.0)

    # Sort DataFrame descending by starred_fit score
    ranked_df = df.sort_values(by="starred_fit", ascending=False)

    return ranked_df

In [ ]:
ranker(df, target_phrases=["aspiring human resources", "seeking human resources"], starred=[23, 41], alpha=0.5)

How can we filter out candidates which in the first place should not be in this list?

    One way would be to clean the data to exclude any candidate that does not contain the words "human resources" in their title. This could be done manually, or it could be built into the code so that it can be run on different sets of data. This model will naturally put inappropriate candidates at the bottom of the list, but will not remove them entirely.

Can we determine a cut-off point that would work for other roles without losing high potential candidates?

    It's likely that you could choose a score below which you would not consider candidates. Looking at this model .30 appears to be a good threshold below which the candidates are not a great fit. The model would have to be tested on multiple different datasets for a different roles to determine if .30 can be used as a universal cutoff.

Do you have any ideas that we should explore so that we can even automate this procedure to prevent human bias?

    Viewing the candidates without personally identifiable information is a good start. It's the same reason why APA formatting uses initials instead of names and many orchestras do blind auditions which the musician is behind a curtain; some orchestras go so far as to include carpet so that assumptions cannot be drawn from the sound of a person's shoes or their gait. Names offer the opportunity for bias. Letting the model rank people for you is a good next step as it applies rules consistently to all candidates. It is important to remember that models can contain bias. For instance, I have built this one to prefer candidates with more LinkedIn connections which might exclude perfectly good candidates who are early in their career; everyone has to start somewhere.